# Sub-Agents

**Prerequisites:** `04_planning.ipynb`, and from `01_foundations`: `01_your_first_llm.ipynb`

`Graph` runs several *tool calls* at once. Every one of them is a single function with known arguments, decided in advance by one planner.

Some work doesn't decompose that way. "Research these two topics and compare them" isn't two tool calls — it's two open-ended jobs, each of which may need several tool calls, a few decisions, and a different number of turns. What that needs isn't a parallel step. It's another agent.

This notebook builds forking: a `Think` that decides to split into sub-agents, each running its own `Runtime` with its own conversation. Then the harder question — what those sub-agents are allowed to see, of the outer conversation and of each other — which turns out to be the difference between delegation, self-consistency, and debate.

**By the end you'll have:**
- Made delegation a decision `Think` reaches through native tool calling, rather than a separate class
- Seen a branch that inherited the outer conversation answer *another* branch's question, and fixed it by giving branches nothing but a self-contained goal
- Split a shared budget across branches, using the `remaining_budget` notebook 2 left unused
- Used `persistent` and `broadcast` as two independent flags, and worked through what each of the four combinations is actually for
- Run the same goal several times at different temperatures and had the outer loop reconcile the answers — chapter 1's sampling knobs used for something new
- Built `SelectThink`, which scores candidate branches and prunes the weak ones, and used fork → select → `repeat_from` as beam search

## Setup

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))
sys.path.append(str(Path.cwd().parent.parent / "01_foundations"))

from chat import Conversation
from runtime import Runtime
from observe import Observe
from think import Think, SelectThink
from act import Act

In [ ]:
from datetime import datetime
from typing import Literal
from zoneinfo import ZoneInfo, ZoneInfoNotFoundError

from pydantic import BaseModel, field_validator

def get_current_time(timezone):
    return datetime.now(ZoneInfo(timezone)).strftime("%Y-%m-%d %H:%M %Z")

def convert_temperature(value, from_unit, to_unit):
    if from_unit.upper() == "F" and to_unit.upper() == "C":
        return round((value - 32) * 5 / 9, 1)
    if from_unit.upper() == "C" and to_unit.upper() == "F":
        return round(value * 9 / 5 + 32, 1)
    raise ValueError(f"unsupported conversion: {from_unit} to {to_unit}")

class GetCurrentTimeArgs(BaseModel):
    timezone: str

    @field_validator("timezone")
    @classmethod
    def timezone_must_be_valid(cls, v):
        try:
            ZoneInfo(v)
        except ZoneInfoNotFoundError:
            raise ValueError(f"'{v}' is not a valid IANA timezone")
        return v

class ConvertTemperatureArgs(BaseModel):
    value: float
    from_unit: Literal["C", "F"]
    to_unit: Literal["C", "F"]

FUNCTIONS = {"get_current_time": get_current_time, "convert_temperature": convert_temperature}
SCHEMAS = {"get_current_time": GetCurrentTimeArgs, "convert_temperature": ConvertTemperatureArgs}
tools = [
    {"type": "function", "function": {
        "name": "get_current_time",
        "description": "Get the current date and time in a given IANA timezone.",
        "parameters": {"type": "object",
                       "properties": {"timezone": {"type": "string", "description": "IANA timezone name"}},
                       "required": ["timezone"]}}},
    {"type": "function", "function": {
        "name": "convert_temperature",
        "description": "Convert a temperature value between Celsius and Fahrenheit.",
        "parameters": {"type": "object",
                       "properties": {"value": {"type": "number"},
                                      "from_unit": {"type": "string", "enum": ["C", "F"]},
                                      "to_unit": {"type": "string", "enum": ["C", "F"]}},
                       "required": ["value", "from_unit", "to_unit"]}}},
]

## Delegation is a tool call

A `Think` that can fork needs to decide *whether* to fork, *how many* branches, and *what each one should do* — and to do that in the same breath as deciding between an ordinary tool call and a final answer. Making that a separate `ForkThink` class would mean a component that only ever forks, declared in a loop position where forking is the only option.

So `delegate` is just another tool in the list:

```python
def _delegate_tool(conversation):
    return {
        "type": "function",
        "function": {
            "name": "delegate",
            "description": (
                "Delegate to independent sub-agents. Use persistent=true, broadcast=true for an "
                "ongoing exchange like a debate, where branches should see each other's turns. "
                "Use persistent=true, broadcast=false for independent specialists you may return "
                "to individually later. Leave both false for a one-off split of work, or for "
                "running the same goal several times with different sampling to check "
                f"consistency (vary temperature/top_p, not the goal, for that).{budget_note}"
            ),
            "parameters": {"type": "object", "properties": {
                "persistent": {"type": "boolean"},
                "broadcast": {"type": "boolean"},
                "branches": {"type": "array", "items": {"type": "object", "properties": {
                    "id": {"type": "string"},
                    "goal": {"type": "string", "description": "A complete, self-contained instruction ..."},
                    "temperature": {"type": "number"},
                    "top_p": {"type": "number"},
                    "max_seconds": {"type": "number"},
                    "max_iterations": {"type": "integer"},
                    ...
                }}},
            }, "required": ["branches"]},
        },
    }
```

`Think.run()` then only needs one extra branch — if the tool it picked is `delegate`, the decision is a fork rather than a tool call:

```python
call = message.tool_calls[0]
if call.function.name == "delegate":
    args = json.loads(call.function.arguments)
    return {"role": "decision", "type": "fork",
            "content": "Forked into: " + "; ".join(b["goal"] for b in args["branches"]),
            "persistent": args.get("persistent", False),
            "broadcast": args.get("broadcast", False),
            "branches": args["branches"]}
```

Everything else — `_build_messages`, the retries, the `respond` path — is untouched. Forking became available to every `Think` in the chapter without a new class.

**`allow_fork` decides who gets offered it.** The tool is appended per call, not per class:

```python
all_tools = [*self.tools, _delegate_tool(conversation)] if self.allow_fork else self.tools
```

That flag exists because of a real failure. A branch given a small concrete job — "get the current time in Tokyo" — would sometimes delegate *that* to a sub-sub-agent. Recursive delegation is occasionally legitimate, but for a leaf task it's pure overhead. Every `Think` in notebooks 2 through 4 passed `allow_fork=False` for this reason; here, the outer one finally gets `True`.

**Budgets are advertised in the description.** `_delegate_tool` is built fresh on every call so it can read `conversation.remaining_budget` — the dict notebook 2 wrote every iteration and nothing used — and name what's actually left:

> The following remain, shared across every branch: 6 iterations, 45 seconds. Split each of these between branches as whole numbers, in proportion to how hard each sub-goal looks.

That's a plain string of numbers `Runtime` already computed. Deciding how to divide it is judgment, which is why it's stated to a `Think` rather than divided by `Act`.

## Running the branches

`Act` handles a `fork` decision by starting each branch in its own thread, each with its own fresh `Runtime`:

```python
def _fork(self, conversation, decision):
    branches = decision["branches"]
    ids = [b.get("id", f"branch{i}") for i, b in enumerate(branches)]

    with concurrent.futures.ThreadPoolExecutor(max_workers=self.max_workers) as pool:
        branch_conversations = list(pool.map(lambda b: self._start_branch(conversation, b), branches))

    results = [{"goal": b["goal"], "id": i, "content": bc.messages[-1]["content"]}
               for b, i, bc in zip(branches, ids, branch_conversations)]
    ...
    return results
```

Same `ThreadPoolExecutor` reasoning as notebook 4, and the same rule as an ordinary tool call: `_fork` doesn't return until every branch is finished. A fork is a blocking call that happens to run several agents inside it.

Where does a branch's `Runtime` come from? `Act` doesn't know what a branch's loop should look like — that's a design decision belonging to whoever declared the outer loop. So `Act` takes a factory:

```python
Act(functions=..., schemas=..., branch_runtime=make_branch_runtime)
```

which receives the branch dict and returns a `Runtime` for it. That's also where the branch's share of the budget and its sampling settings get applied:

In [ ]:
def make_branch_runtime(branch):
    return Runtime(
        loop=[
            Think(tools=tools, allow_fork=False,
                  temperature=branch.get("temperature"), top_p=branch.get("top_p")),
            Act(functions=FUNCTIONS, schemas=SCHEMAS),
            Observe(),
        ],
        repeat_from=0,
        max_iterations=branch.get("max_iterations") or 9,
        max_seconds=branch.get("max_seconds"),
        max_tool_calls=branch.get("max_tool_calls"),
        max_tokens=branch.get("max_tokens"),
    )

A branch is an ordinary ReAct loop — notebook 2's, unchanged. `temperature` and `top_p` come straight from chapter 1's notebook 1, threaded through `Think` into `client.chat.completions.create()`; they'll matter later in this notebook.

`max_iterations` falls back to 9 when the delegating `Think` didn't assign one, so a branch is never unbounded. And because branch budgets come from a dict the model filled in, a branch is protected by notebook 2's machinery regardless of what it was handed.

## What a branch is allowed to see

Now the question that decides whether any of this works.

A sub-agent needs context. The obvious way to give it context is to hand it the conversation so far — it's right there, and it contains everything relevant.

Here is what that does. The outer request has two parts; the branch has been assigned one of them:

In [ ]:
OUTER_REQUEST = "What's the current time in Tokyo, and what's 100 degrees Fahrenheit in Celsius?"
BRANCH_GOAL = "Get the current time in Tokyo."

outer = Conversation()
outer.messages.append({"role": "user", "content": OUTER_REQUEST})

# the tempting design: seed the branch with everything the outer agent knows
leaky = Conversation()
leaky.messages = list(outer.messages) + [{"role": "user", "content": BRANCH_GOAL}]
make_branch_runtime({}).run(leaky)
print("inherited history ->", leaky.messages[-1]["content"])

In [ ]:
# what Act actually does: system messages (if any), plus the branch's own goal, and nothing else
isolated = Conversation()
isolated.messages = [{"role": "user", "content": BRANCH_GOAL}]
make_branch_runtime({}).run(isolated)
print("goal only        ->", isolated.messages[-1]["content"])

The branch that inherited the conversation answered the temperature question too — a question that was never its job, that another branch was simultaneously working on, and whose answer will now be reported twice to the outer loop as if two things had been confirmed independently.

It isn't confused. It's doing exactly what the context asks: the original two-part request is sitting right there in its history, so it answers both parts.

```python
def _start_branch(self, conversation, branch):
    branch_conversation = Conversation()
    branch_conversation.messages = [m for m in conversation.messages if m["role"] == "system"]
    branch_conversation.messages.append({"role": "user", "content": branch["goal"]})
    self.branch_runtime(branch).run(branch_conversation)
    return branch_conversation
```

A branch gets system messages, if any, and its own goal. Nothing else. Not the outer request, not the other branches' goals, not the reasoning that led to the split.

That pushes the burden onto the goal itself, which is why `_delegate_tool`'s description of that field is so insistent:

> A complete, self-contained instruction for an agent with NO other context — it will not see this conversation or any other branch's goal. Include everything it actually needs (e.g. the specific question or values involved). Never write it as if the reader already knows the broader request, and never reference 'the other task' or another branch.

Writing a good goal *is* the delegation. A `Think` that writes "handle the second part" has not delegated anything, and the isolation makes that failure loud and immediate rather than subtle.

## Decomposition: the one-shot fork

Both flags off. Branches run once, report back, and are discarded:

In [ ]:
decompose = Runtime(
    loop=[
        Observe(event=OUTER_REQUEST),
        Think(tools=tools, allow_fork=True),
        Act(functions=FUNCTIONS, schemas=SCHEMAS, branch_runtime=make_branch_runtime),
        Observe(),
        Think(tools=tools, allow_fork=False),
        Act(functions=FUNCTIONS, schemas=SCHEMAS),
    ],
    max_iterations=12,
)

result = decompose.run(Conversation())
for m in result.messages:
    print(f"- {m['role']:9} {m['content']!r}")

The `fork` decision names the goals it invented; `Act` ran them concurrently; `Observe` merged the results into a single `tool` observation, one line per branch:

```python
if isinstance(effect, list):
    content = "\n".join(f"[{b.get('goal') or b.get('id')}] {b['content']}" for b in effect)
    return {"role": "tool", "content": content}
```

This is the join. Notebook 2's `Observe` handled one effect; a fork produces a list, and the merge is a labelled concatenation — mechanical, with the labels preserving which branch said what so the next `Think` can attribute claims correctly.

Note this loop has no `repeat_from` — it's a straight sequence, because forking once and answering is genuinely a pipeline. Nothing forced that; it's what was declared.

## Self-consistency: same goal, different sampling

Every branch so far got a different goal. Nothing requires that.

Give several branches the *same* goal and vary `temperature` and `top_p`, and the fork stops being about dividing work and starts being about disagreement: run the reasoning several times independently, and see whether it lands in the same place. Where it doesn't, you've learned the answer wasn't stable.

Chapter 1's notebook 1 introduced those knobs to control a single answer. Here they're the mechanism for generating *several*, deliberately varied, from one model.

A question models often get wrong in a specific, tempting way:

In [ ]:
TRICK = ("I have 3 apples today. Yesterday I ate 1 apple. How many apples do I have today? "
         "Answer with a number and one short sentence of reasoning.")

consistency = Runtime(
    loop=[
        Observe(event=(
            f"{TRICK}\n\n"
            "Delegate this exact question to 3 independent branches, each with the identical goal "
            "but a different temperature (try 0.2, 0.8, and 1.2). Do not change the wording between "
            "branches. Then report which answer the branches agreed on."
        )),
        Think(allow_fork=True),
        Act(branch_runtime=make_branch_runtime),
        Observe(),
        Think(allow_fork=False),
        Act(),
    ],
    max_iterations=12,
)

voted = consistency.run(Conversation())
for m in voted.messages:
    print(f"- {m['role']:9} {m['content']!r}\n")

Three runs of one question, sampled differently, then a final `Think` reading all three at once and reconciling them.

Two things are worth noticing about how little machinery this took.

**There is no voting component.** The merged observation contains every branch's answer, labelled. A `Think` reading three labelled answers and picking the one that recurs is just a `Think` reading its context. Adding a `VoteThink` would have hardcoded one reconciliation policy — majority — when "two agree but the third gives a better reason" is a judgment worth keeping available.

**The model is the same across branches; only sampling differs.** Different models would confound the measurement: a disagreement could mean the reasoning is unstable, or just that two models differ. Holding the model fixed keeps the answer to one question.

## `persistent` and `broadcast`

Both demos so far ran branches once and threw them away. Two flags change that, and they are deliberately *not* one `mode` field, because all four combinations are things you'd want.

```python
if broadcast:
    for i, bc in zip(ids, branch_conversations):
        for other_id, result in zip(ids, results):
            if other_id != i:
                bc.messages.append({"role": "user", "content": f'[{other_id}] said: {result["content"]}'})

if persistent:
    conversation.branches = dict(zip(ids, branch_conversations))
    conversation.branches_broadcast = broadcast
```

|  | `broadcast=False` | `broadcast=True` |
|---|---|---|
| **`persistent=False`** | Decomposition, self-consistency — the two demos above | Branches see each other once, then end. Rarely what you want, but harmless |
| **`persistent=True`** | Independent specialists you can return to individually, each keeping its own accumulated context | Debate — branches keep going *and* see each other's turns |

**`persistent`** keeps each branch's `Conversation` alive on `conversation.branches`, so it can be resumed later with its own history intact rather than restarted from its goal.

**`broadcast`** copies each branch's result into every sibling's history. Two details in that loop were both bugs first:

- `results` is captured *before* the broadcast loop runs. Reading `bc.messages[-1]` afterwards returns whatever was just appended to it — a sibling's message — so each branch was reported as having said what the other one said.
- The copy goes in as `{"role": "user", "content": "[other] said: ..."}`, not under the branch's own role. Appended as its own turn, a branch's next call reads the opponent's argument as something it said itself, and promptly agrees with it.

Both are the same underlying mistake: forgetting that a branch's conversation is written from *its* point of view.

### Resuming a branch

```python
def _advance_branch(self, conversation, decision):
    branch_conversation = conversation.branches[decision["branch"]]
    if decision.get("message"):
        branch_conversation.messages.append({"role": "user", "content": decision["message"]})
    self.branch_runtime({"goal": None}).run(branch_conversation)
    content = branch_conversation.messages[-1]["content"]

    if getattr(conversation, "branches_broadcast", False):
        for other_id, other_conversation in conversation.branches.items():
            if other_id != decision["branch"]:
                other_conversation.messages.append({"role": "user", "content": f"[{decision['branch']}] said: {content}"})

    return [{"goal": None, "id": decision["branch"], "content": content}]
```

A fresh `Runtime` over an existing conversation — the branch's accumulated history is what makes it a continuation rather than a restart. A turn is a full run of the branch's own loop, not one message: responding to an argument may take several tool calls.

**An honest gap.** `Act` implements `advance_branch`, but no `Think` currently *decides* it — there's no synthetic tool for it the way `delegate` has one, so the demo below issues it by hand.

That's not an oversight so much as an unresolved design question. With two debaters, alternating is obvious. With four, deciding who speaks next is real judgment — and a round-robin schedule would be a policy baked into `Act`, exactly the kind of decision this chapter keeps insisting belongs to `Think`. Adding an `advance` tool alongside `delegate` is the natural fix; choosing the turn-taking policy first is the actual work.

In [ ]:
debate = Runtime(
    loop=[
        Observe(event=(
            "Should a small startup write its backend in Python or in Go? Set up a debate: delegate "
            "to exactly 2 persistent branches that broadcast to each other. Give one the id 'python' "
            "and the goal of arguing the case for Python, and the other the id 'go' and the goal of "
            "arguing the case for Go. Each goal must be self-contained and state the question."
        )),
        Think(allow_fork=True),
        Act(branch_runtime=make_branch_runtime),
        Observe(),
    ],
    max_iterations=8,
)

arena = debate.run(Conversation())
for branch_id, branch_conversation in arena.branches.items():
    print(f"[{branch_id}] opening: {branch_conversation.messages[1]['content'][:160]}...")
print()
print("branches kept alive:", list(arena.branches), "| broadcasting:", arena.branches_broadcast)

Both branches ran, and both are still alive on `arena.branches`. Because `broadcast=True`, each one's history now also contains the other's opening argument, attributed.

Now give one of them another turn — the `advance_branch` decision issued by hand, since nothing decides it yet:

In [ ]:
arena.messages.append({
    "role": "decision", "type": "advance_branch", "branch": "python",
    "content": "python responds to go's argument",
    "message": "Respond directly to the strongest point the other side just made.",
})

act = Act(branch_runtime=make_branch_runtime)
act.run(arena)
rebuttal = Observe().run(arena)
print("python's rebuttal:\n", rebuttal["content"][:600])

The rebuttal engages with what Go actually argued, because Go's argument is in Python's history — as a `user` turn labelled `[go] said:`, not as something Python said itself.

Notice that the debate branches were created with `Think(allow_fork=True)` reading a request that described the setup. Nothing about "debate" exists in the code. It's `persistent=True, broadcast=True` plus goals that happen to take opposing positions.

## Choosing between branches: `SelectThink`

`Observe` merges every branch result into the conversation. When branches divided up work, keeping all of them is right — each holds a different piece.

When branches are *competing* — several attempts at the same thing — keeping all of them is waste. Every future turn re-reads candidates already known to be worse.

```python
class Selection(BaseModel):
    keep_indices: list[int]
    reasoning: str

class SelectThink(Think):
    def __init__(self, keep=1):
        self.keep = keep

    def run(self, conversation):
        candidates = getattr(conversation, "pending", None) or []
        listing = "\n".join(f"{i}. [{c.get('goal', '(no goal)')}] {c['content']}" for i, c in enumerate(candidates))
        result = extract(
            f"Conversation so far:\n{render(conversation)}\n\n"
            f"Candidate results from {len(candidates)} independent branches:\n{listing}\n\n"
            f"Score each against the original goal, and pick the best {self.keep} to continue "
            "exploring, by their index in the list above. Discard the rest.\n"
            f"Return JSON matching this schema:\n{Selection.model_json_schema()}",
            Selection,
        )
        valid = [i for i in dict.fromkeys(result.keep_indices) if 0 <= i < len(candidates)]
        return {"role": "decision", "type": "prune", "content": result.reasoning,
                "keep_indices": valid[: self.keep] or list(range(min(self.keep, len(candidates))))}
```

**That last line is load-bearing, and it was a bug first.** The prompt originally ended "Return JSON matching the given schema" without ever including the schema. Asked to return JSON with no field names given, the model reasonably invented its own — a list of scored candidates under a key like `taglines` — and `Selection` rejected every one. It sometimes recovered, because `extract()` feeds each validation error back and tries again, but "sometimes" is not a design. `DebugThink` got this right from the start by embedding `model_json_schema()`; this now does the same.

`SelectThink` reads `conversation.pending` — the raw branch results, *before* `Observe` has merged them — and emits a `prune` decision. `Act` applies it, and the application is one line:

```python
if decision["type"] == "prune":
    pending = conversation.pending or []
    conversation.pending = [pending[i] for i in decision["keep_indices"]]
    return None
```

An index filter. All the judgment happened in `SelectThink`; `Act` executes the list it was handed. So `SelectThink` has to sit between the fork's `Act` and the `Observe` that would merge — pruning after the merge would mean the discarded candidates are already in `messages`, which is the cost the pruning exists to avoid.

Because `Act` indexes straight into `pending`, the indices had better be real ones, and the model does not reliably supply that. It returns duplicates, more than it was asked for, and — as this notebook found on its first run, with an `IndexError` — indices that don't exist at all. So the guarantee is made in `SelectThink`, which is the only place the candidate list is in scope: deduplicate, drop anything out of range, truncate to `keep`, and fall back to the first `keep` candidates if nothing usable survives. Validating in `Act` instead would mean `Act` reasoning about a decision it was handed, which is the one thing it doesn't do.

## Beam search

`keep=1` narrows to a single winner. `keep=2` or more keeps a *frontier*, and then the loop can expand what survived — which is beam search, and by extension Tree of Thoughts:

```
Observe(event=problem)
Think(allow_fork=True)   -> fork into k candidate approaches
Act(branch_runtime=...)  -> run them, pending = k results
SelectThink(keep=b)      -> score them, prune to the best b
Act()                    -> apply the prune
Observe()                -> merge the survivors into the conversation
                            repeat_from sends control back to Think, which
                            now forks again *from the survivors*
```

Nothing in that sequence is new. It's the fork machinery, `SelectThink`, and `repeat_from` — three things that already existed, in an order nobody had to add support for.

In [ ]:
beam = Runtime(
    loop=[
        Observe(event=(
            "Come up with a memorable tagline for a new brand of running shoes. Delegate to 3 "
            "branches, each writing one candidate tagline in a distinctly different style. Give "
            "each branch a self-contained goal that states the product. When you are given "
            "surviving candidates, either delegate again to refine them further or, if one is "
            "clearly strong, give it as your final answer."
        )),
        Think(allow_fork=True),
        Act(branch_runtime=make_branch_runtime),
        SelectThink(keep=2),
        Act(),
        Observe(),
    ],
    repeat_from=1,
    max_iterations=16,
)

searched = beam.run(Conversation())
for m in searched.messages:
    print(f"- {m['role']:9} {m['content']!r}\n")

Each round generates candidates, scores them, discards the weak ones, and reports only the survivors — which the next `Think` reads and expands. The pruned candidates never entered `messages` at all, so the context holds the frontier rather than the whole search tree.

**What this doesn't do: backtrack.** Once a candidate is pruned, it's gone from `pending` and survives only as whatever the merged observation happened to say. Classic Tree of Thoughts can return to a branch it abandoned; this can't. Keeping every candidate on the conversation, scored, and letting a `Think` reopen one is the obvious extension — and it's a bookkeeping change, not a structural one.

## What you built

✓ Made forking a decision reached through the normal tool-calling path — `delegate` as one more tool — so every `Think` can fork, gated per instance by `allow_fork` after leaf branches were caught delegating trivial work further

✓ Watched a branch seeded with the outer conversation answer a sibling's question, and fixed it by giving branches only a system message plus a self-contained goal — pushing the work onto the goal, where writing it well *is* the delegation

✓ Split the shared budget across branches by stating what remains in the `delegate` tool's own description — the `remaining_budget` notebook 2 computed and left unused

✓ Ran decomposition (both flags off) and self-consistency (same goal, three temperatures, chapter 1's knobs used to generate disagreement rather than suppress it), with the outer `Think` reconciling results and no voting component needed

✓ Worked through `persistent` × `broadcast` as four distinct configurations, and ran a real debate — two branches that keep their history and see each other's turns, attributed as the other's words rather than their own

✓ Built `SelectThink`, scoring branch results in `pending` before `Observe` can merge them, so pruned candidates never cost context

✓ Assembled beam search — fork, select, `repeat_from` — out of three mechanisms that already existed, and named the one thing it genuinely can't do: return to a candidate it discarded

**Next:** `06_agent_patterns.ipynb` — the whole catalogue, every pattern written as a declared loop over the components these five notebooks built.